# SSAEP + SSVEP Combined Migraine Detection (self-contained, late fusion)

A subject-level classifier that **fuses the SSAEP and SSVEP conditions**.

This notebook is **self-contained**: all code (window building, the 42
features, PCA, the 8-model zoo, LOSO and late-fusion evaluation) is inlined.

**What "late fusion" means here:**
- For each subject and each fold of **Leave-One-Subject-Out**, two condition
  models are trained -- one on SSAEP windows, one on SSVEP windows (each on
  data from all *other* subjects of that condition).
- Both models are applied to the held-out subject's own condition windows.
- The subject's final score = mean of **all per-window probabilities** from
  whichever condition(s) the subject actually has.

**Conditional availability is handled automatically:** a subject missing one
condition (e.g. `M13` has SSVEP only) simply contributes through the condition
it does have. A subject is excluded only if it has neither condition.

**Exclusions (same as the resting baseline):**
`M2`, `M6`, `M18` (medicated) and `C2`, `C6`, `C12` (unmatched extra controls).


## 1. Setup and data loading

In [ ]:
# ============================================================================
# 1. SETUP - paths, EEG constants, and data loading
# ============================================================================

import os
import re
import glob
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import signal as sp_signal
from scipy.stats import entropy as sp_entropy

warnings.filterwarnings('ignore')

# --- Folders ----------------------------------------------------------------
PROJECT_ROOT = Path.cwd()                          # notebooks live at project root
PREPROCESSED = PROJECT_ROOT / 'data' / 'MIGRAINE_GPU_preprocessed'
DEMOGRAPHICS = PROJECT_ROOT / 'Dataset' / 'Migraine_Control_Demographics.xlsx'
OUTPUT_DIR   = PROJECT_ROOT / 'output'
OUTPUT_DIR.mkdir(exist_ok=True)

# --- EEG constants ------------------------------------------------------------
SFREQ        = 512.0      # sampling rate (Hz) of the saved epochs
EPOCH_S      = 2.0        # one saved epoch is 2 seconds long
WINDOW_EPOCHS = 10        # we group 10 contiguous epochs into a 20-second window
RANDOM_STATE = 42         # reproducible random numbers

# --- Subjects excluded from the analysis ------------------------------------
EXCLUDES       = {'M2', 'M6', 'M18'}    # took medication before the recording
EXTRA_CONTROLS = {'C2', 'C6', 'C12'}    # unmatched extra controls

# --- EEG frequency bands used for the band-power features -------------------
BANDS = {
    'delta': (0.5, 4.0),
    'theta': (4.0, 8.0),
    'alpha': (8.0, 13.0),
    'beta' : (13.0, 30.0),
    'gamma': (30.0, 100.0),
}

# 32-channel 10-20 subset used for the connectivity (wPLI) features
CONN_CHANNELS = [
    'Fp1', 'Fp2', 'AF7', 'AF8',
    'F7', 'F3', 'Fz', 'F4', 'F8',
    'FC5', 'FC1', 'FC2', 'FC6',
    'T7', 'C3', 'Cz', 'C4', 'T8',
    'CP5', 'CP1', 'CP2', 'CP6',
    'P7', 'P3', 'Pz', 'P4', 'P8',
    'PO9', 'O1', 'Oz', 'O2', 'PO10',
]


def load_labels():
    """Return {subject_id: 1 if migraine, 0 if control} from the Excel file."""
    df = pd.read_excel(DEMOGRAPHICS)
    labels = {}
    for _, row in df.iterrows():
        pid = str(row['P#']).strip()
        labels[pid] = 0 if pid.startswith('C') else 1
    return labels


def discover_files(condition):
    """Find every '*_broadband.npy' file inside a condition folder.

    Returns (folder, records) where each record holds the file path plus the
    recording-folder name and the base subject id parsed from the filename.
    """
    cond_dir = PREPROCESSED / condition
    files = sorted(glob.glob(str(cond_dir / '*_broadband.npy')))
    records = []
    for f in files:
        name = os.path.basename(f)
        #  e.g.  C1_C1_SSAEP_broadband.npy   or   M15_2_P15_SSVEP_broadband.npy
        m = re.match(r'^(C\d+|M\d+_\d+)_(.+)_broadband\.npy$', name)
        if not m:
            continue
        subject_dir = m.group(1)                  # the recording folder name
        base_id = subject_dir.split('_')[0]       # base subject id (e.g. M15)
        records.append({'file': f, 'subject_dir': subject_dir, 'base_id': base_id})
    return cond_dir, records


def channel_names():
    """Read the ordered list of EEG channel names (130 channels)."""
    with open(PREPROCESSED / 'channel_names.txt') as fh:
        return [line.strip() for line in fh if line.strip()]


## 2. Feature extraction helpers (42 features per 20-s window)

In [ ]:
# ============================================================================
# 2. FEATURE EXTRACTION - 42 features per 20-second window
# ============================================================================
# Spectral power (Welch), then band power / alpha peak / spectral entropy,
# plus wPLI connectivity and topographic asymmetry.


def welch_psd(data, sfreq=SFREQ):
    """Power spectral density via Welch's method (averaged per channel)."""
    freqs, psd = sp_signal.welch(
        data, fs=sfreq, nperseg=int(4 * sfreq), noverlap=int(2 * sfreq), axis=-1)
    return freqs, psd


def band_power_features(psd, freqs):
    """Relative power in each band = band area / total spectrum area."""
    total = np.trapezoid(psd, freqs, axis=-1)
    total[total == 0] = 1e-12
    out = []
    for _, (lo, hi) in BANDS.items():
        mask = (freqs >= lo) & (freqs <= hi)
        bp = np.trapezoid(psd[:, mask], freqs[mask], axis=-1)
        out.append(bp / total)
    return np.stack(out, axis=-1)            # shape (n_channels, n_bands)


def alpha_peak_frequency(psd, freqs):
    """Frequency (Hz) of the strongest peak inside the 8-13 Hz alpha band."""
    mask = (freqs >= 8.0) & (freqs <= 13.0)
    return freqs[mask][np.argmax(psd[:, mask], axis=-1)]


def spectral_entropy_features(psd):
    """Normalised spectral entropy per channel (0 = flat, 1 = peaky)."""
    p = psd / (psd.sum(axis=-1, keepdims=True) + 1e-12)
    return sp_entropy(p, axis=-1) / np.log(psd.shape[-1])


def wpli_matrix(data):
    """Weighted Phase Lag Index between every pair of channels (n_ch, n_ch)."""
    n_ch = data.shape[0]
    phase = np.angle(sp_signal.hilbert(data, axis=-1))
    wpli = np.zeros((n_ch, n_ch))
    for i in range(n_ch):
        for j in range(i + 1, n_ch):
            imag = np.sin(phase[i] - phase[j])
            num = np.abs(np.mean(imag))
            den = np.mean(np.abs(imag))
            wpli[i, j] = wpli[j, i] = num / (den + 1e-12)
    return wpli


def connectivity_graph_features(wpli):
    """Summarise the wPLI graph with a small set of scalar statistics."""
    n = wpli.shape[0]
    triu = wpli[np.triu_indices(n, k=1)]
    feats = {
        'conn_mean':   triu.mean(),
        'conn_std':    triu.std(),
        'conn_max':    triu.max(),
        'conn_min':    triu.min(),
        'conn_median': np.median(triu),
    }
    # clustering coefficient: how strongly each channel's neighbours connect
    w = wpli.copy()
    np.fill_diagonal(w, 0)
    clustering = np.zeros(n)
    for i in range(n):
        neighbours = np.where(w[i] > 0)[0]
        if len(neighbours) < 2:
            continue
        sub = w[np.ix_(neighbours, neighbours)]
        clustering[i] = sub.sum() / (len(neighbours) * (len(neighbours) - 1) + 1e-12)
    feats['conn_clustering'] = clustering.mean()
    return feats
def topographic_asymmetry(band_power, ch_names):
    """Left/right and anterior/posterior power ratios for each band."""
    idx = {ch: i for i, ch in enumerate(ch_names)}
    left  = ['Fp1', 'F7', 'F3', 'T7', 'C3', 'P7', 'P3', 'O1']
    right = ['Fp2', 'F8', 'F4', 'T8', 'C4', 'P8', 'P4', 'O2']
    ant   = ['Fp1', 'Fp2', 'F7', 'F3', 'Fz', 'F4', 'F8']
    post  = ['P7', 'P3', 'Pz', 'P4', 'P8', 'O1', 'O2']
    feats = {}
    for b, band_name in enumerate(BANDS):
        lr = np.mean([band_power[idx[c], b] for c in left if c in idx]) / \
             (np.mean([band_power[idx[c], b] for c in right if c in idx]) + 1e-12)
        ap = np.mean([band_power[idx[c], b] for c in ant if c in idx]) / \
             (np.mean([band_power[idx[c], b] for c in post if c in idx]) + 1e-12)
        feats[f'LR_{band_name}'] = lr
        feats[f'AP_{band_name}'] = ap
    return feats


def extract_window_features(window_data, ch_names):
    """Turn one 20-second window (channels x time) into a dict of 42 features."""
    feats = {}
    freqs, psd = welch_psd(window_data)
    bp  = band_power_features(psd, freqs)          # (n_ch, 5)
    apf = alpha_peak_frequency(psd, freqs)
    se  = spectral_entropy_features(psd)

    # 1) Relative band power: mean/std/max/min across channels (log-scaled)
    for b, band_name in enumerate(BANDS):
        feats[f'bp_{band_name}_mean'] = np.log1p(bp[:, b].mean())
        feats[f'bp_{band_name}_std']  = np.log1p(bp[:, b].std())
        feats[f'bp_{band_name}_max']  = np.log1p(bp[:, b].max())
        feats[f'bp_{band_name}_min']  = np.log1p(bp[:, b].min())

    # 2) Alpha peak frequency and spectral entropy: channel summaries
    feats['apf_mean'] = apf.mean(); feats['apf_std'] = apf.std()
    feats['apf_max']  = apf.max();  feats['apf_min'] = apf.min()
    feats['se_mean']  = se.mean();  feats['se_std']  = se.std()

    # 3) wPLI connectivity on the 32-channel subset (if enough channels exist)
    conn_idx = [ch_names.index(c) for c in CONN_CHANNELS if c in ch_names]
    if len(conn_idx) >= 16:
        feats.update(connectivity_graph_features(wpli_matrix(window_data[conn_idx])))

    # 4) Topographic asymmetry ratios
    feats.update(topographic_asymmetry(bp, ch_names))
    return feats


def build_dataset(condition):
    """Extract per-window features for one condition.

    Returns: X (n_windows x n_features), y (labels), groups (subject per window),
    feat_names, subject_stats (DataFrame), n_skipped (dropped recordings).
    """
    _, records = discover_files(condition)
    labels = load_labels()
    chs = channel_names()

    X_list, y_list, groups_list = [], [], []
    feat_names, subject_stats = None, []
    n_skipped = 0

    for r in records:
        base = r['base_id']
        # (1) drop excluded / unknown subjects
        if base in EXCLUDES or base in EXTRA_CONTROLS or base not in labels:
            n_skipped += 1
            continue
        arr = np.load(r['file'])                   # (n_epochs, n_ch, n_time)
        if arr.shape[0] == 0:
            n_skipped += 1
            continue
        # (2) group contiguous epochs into 20-second windows
        n_windows = arr.shape[0] // WINDOW_EPOCHS
        if n_windows == 0:
            n_skipped += 1
            continue
        for w in range(n_windows):
            start, end = w * WINDOW_EPOCHS, (w + 1) * WINDOW_EPOCHS
            # reshape (10, n_ch, n_time) -> (n_ch, 10*n_time)
            window_data = arr[start:end].transpose(1, 0, 2).reshape(arr.shape[1], -1)
            feats = extract_window_features(window_data, chs)
            if feat_names is None:
                feat_names = list(feats.keys())
            X_list.append(list(feats.values()))
            y_list.append(labels[base])
            groups_list.append(r['subject_dir'])
        # (3) remember per-subject bookkeeping
        subject_stats.append({
            'subject': r['subject_dir'],
            'group': 'migraine' if labels[base] == 1 else 'control',
            'n_epochs': int(arr.shape[0]),
            'n_windows': n_windows,
        })

    X = np.array(X_list, dtype=np.float64)
    y = np.array(y_list)
    groups = np.array(groups_list)
    return X, y, groups, feat_names, pd.DataFrame(subject_stats), n_skipped


## 3. Models, evaluation, and late-fusion helpers

In [ ]:
# ============================================================================
# 3. MODELS AND EVALUATION HELPERS
# ============================================================================

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (RandomForestClassifier, ExtraTreesClassifier,
                              GradientBoostingClassifier)
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPClassifier

from sklearn.model_selection import LeaveOneGroupOut, GroupKFold
from sklearn.metrics import (roc_auc_score, accuracy_score, precision_score,
                             recall_score, f1_score, confusion_matrix, roc_curve)

# 8 classic classifiers stored as 'builders' (callables). Each is a fresh
# instance we can retrain inside every cross-validation fold.
MODEL_ZOO = {
    'LogisticRegression': lambda: LogisticRegression(C=1.0, max_iter=3000,
                                                     random_state=RANDOM_STATE),
    'RandomForest': lambda: RandomForestClassifier(n_estimators=300, max_depth=8,
                                                   random_state=RANDOM_STATE),
    'ExtraTrees': lambda: ExtraTreesClassifier(n_estimators=300, max_depth=8,
                                               random_state=RANDOM_STATE),
    'GradientBoosting': lambda: GradientBoostingClassifier(n_estimators=200,
                                        learning_rate=0.05, max_depth=3,
                                        random_state=RANDOM_STATE),
    'SVM_RBF': lambda: SVC(C=1.0, kernel='rbf', gamma='scale', probability=True,
                           random_state=RANDOM_STATE),
    'KNN': lambda: KNeighborsClassifier(n_neighbors=7),
    'GaussianNB': lambda: GaussianNB(),
    'MLP': lambda: MLPClassifier(hidden_layer_sizes=(64,), max_iter=800,
                                 early_stopping=True, random_state=RANDOM_STATE),
}


def standardize_pca(X):
    """Standardise features (zero mean, unit variance) then apply PCA."""
    scaler = StandardScaler()
    Xs = scaler.fit_transform(X)
    n_comp = min(50, X.shape[1], X.shape[0])
    pca = PCA(n_components=n_comp, random_state=RANDOM_STATE)
    Xp = pca.fit_transform(Xs)
    return Xp, scaler, pca, float(pca.explained_variance_ratio_.sum())
def evaluate_metrics(prob, true):
    """AUC + accuracy/precision/recall/F1 at the Youden-optimal threshold."""
    fpr, tpr, thr = roc_curve(true, prob)
    best_thr = float(thr[np.argmax(tpr - fpr)])   # maximises sensitivity + specificity
    pred = (prob >= best_thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(true, pred).ravel()
    return {
        'subject_auc':      float(roc_auc_score(true, prob)),
        'subject_accuracy': float(accuracy_score(true, pred)),
        'subject_precision': float(precision_score(true, pred, zero_division=0)),
        'subject_recall':   float(recall_score(true, pred, zero_division=0)),
        'subject_f1':       float(f1_score(true, pred, zero_division=0)),
        'youden_threshold': best_thr,
        'cm': [[int(tn), int(fp)], [int(fn), int(tp)]],
        'roc': (fpr, tpr),
    }


def evaluate_loso(builder, X, y, groups):
    """Leave-One-Subject-Out: every subject is the test set exactly once.

    Each window keeps its subject id (groups), so no subject's data leaks into
    the training set. A subject's final score = mean prob over its windows.
    """
    logo = LeaveOneGroupOut()
    fold_aucs = []
    subj_prob, subj_true = {}, {}
    for tr, te in logo.split(X, y, groups):
        clf = builder()
        clf.fit(X[tr], y[tr])
        prob = clf.predict_proba(X[te])[:, 1]
        fold_aucs.append(roc_auc_score(y[te], prob))
        grp = groups[te][0]
        subj_prob.setdefault(grp, []).append(prob)
        subj_true[grp] = y[te][0]

    order = sorted(subj_prob)
    prob_arr = np.array([np.mean(np.concatenate(subj_prob[g])) for g in order])
    true_arr = np.array([subj_true[g] for g in order])

    res = evaluate_metrics(prob_arr, true_arr)
    res['window_auc'] = float(np.mean(fold_aucs))
    res['window_auc_std'] = float(np.std(fold_aucs))
    res['n_subjects'] = len(order)
    res['n_migraine'] = int(true_arr.sum())
    res['subjects'] = order
    res['subject_probs'] = prob_arr
    res['subject_true'] = true_arr
    return res


def grouped_window_auc(builder, X, y, groups, splits=5):
    """Window-level AUC via grouped 5-fold CV (each fold holds out whole subjects)."""
    gkf = GroupKFold(n_splits=splits)
    aucs = []
    for tr, te in gkf.split(X, y, groups):
        clf = builder()
        clf.fit(X[tr], y[tr])
        prob = clf.predict_proba(X[te])[:, 1]
        try:
            aucs.append(roc_auc_score(y[te], prob))
        except Exception:
            continue
    if not aucs:
        return float('nan'), float('nan')
    return float(np.mean(aucs)), float(np.std(aucs))


def run_all_models(X, y, groups):
    """Run every model in MODEL_ZOO -> (results DataFrame, ROC data dict)."""
    rows, roc_data = [], {}
    for name, builder in MODEL_ZOO.items():
        res = evaluate_loso(builder, X, y, groups)
        wa, wastd = grouped_window_auc(builder, X, y, groups)
        fpr, tpr = res['roc']
        rows.append({
            'model': name,
            'window_auc': wa, 'window_auc_std': wastd,
            'subject_auc': res['subject_auc'],
            'subject_accuracy': res['subject_accuracy'],
            'subject_precision': res['subject_precision'],
            'subject_recall': res['subject_recall'],
            'subject_f1': res['subject_f1'],
            'youden_threshold': res['youden_threshold'],
            'n_subjects': res['n_subjects'], 'n_migraine': res['n_migraine'],
            'cm_tn': res['cm'][0][0], 'cm_fp': res['cm'][0][1],
            'cm_fn': res['cm'][1][0], 'cm_tp': res['cm'][1][1],
        })
        roc_data[name] = (fpr, tpr, res['subject_auc'])
    return pd.DataFrame(rows), roc_data




In [ ]:
def evaluate_late_fusion(builder, X1, y1, g1, X2, y2, g2):
    """Late fusion: train one model per condition, average per-window probs.

    For each held-out subject, build two models (one per condition) on all
    *other* subjects, apply them to this subject's own windows, and use the
    mean probability. A subject missing one condition (e.g. M13) simply
    contributes through the condition it does have.
    """
    uniq1, uniq2 = set(g1), set(g2)
    all_subjects = sorted(uniq1 | uniq2)

    # ground-truth label per subject
    label_of = {}
    for g, yv in zip(g1, y1):
        label_of.setdefault(g, int(yv))
    for g, yv in zip(g2, y2):
        label_of.setdefault(g, int(yv))

    subj_prob = {}
    for test in all_subjects:
        window_probs = []
        if test in uniq1:                          # subject has SSAEP
            tr = g1 != test
            clf = builder(); clf.fit(X1[tr], y1[tr])
            window_probs.append(clf.predict_proba(X1[g1 == test])[:, 1])
        if test in uniq2:                          # subject has SSVEP
            tr = g2 != test
            clf = builder(); clf.fit(X2[tr], y2[tr])
            window_probs.append(clf.predict_proba(X2[g2 == test])[:, 1])
        subj_prob[test] = float(np.mean(np.concatenate(window_probs)))

    order = sorted(subj_prob)
    prob_arr = np.array([subj_prob[g] for g in order])
    true_arr = np.array([label_of[g] for g in order])

    res = evaluate_metrics(prob_arr, true_arr)
    res['n_subjects'] = len(order)
    res['n_migraine'] = int(true_arr.sum())
    res['subjects'] = order
    res['subject_probs'] = prob_arr
    res['subject_true'] = true_arr
    res['conditions'] = 'ssaep+ssvep'
    return res


## 4. Build both conditions and reduce with PCA

In [ ]:
# --- Build both conditions and reduce with PCA ------------------------------
COND_A, COND_B = 'ssaep', 'ssvep'
Xa, ya, ga, fn_a, stats_a, sk_a = build_dataset(COND_A)
Xb, yb, gb, fn_b, stats_b, sk_b = build_dataset(COND_B)
assert fn_a == fn_b, 'feature names differ between conditions'
feat_names = fn_a

Xa_r, sa, pa, va = standardize_pca(Xa)
Xb_r, sb, pb, vb = standardize_pca(Xb)

print(f'SSAEP : {Xa.shape} -> {Xa_r.shape}  subjects={len(stats_a)}  PCAvar={va:.3f}')
print(f'SSVEP : {Xb.shape} -> {Xb_r.shape}  subjects={len(stats_b)}  PCAvar={vb:.3f}')
print(f'SSAEP subjects: {sorted(np.unique(ga))}')
print(f'SSVEP subjects: {sorted(np.unique(gb))}')


## 5. Individual-condition LOSO (reference)

In [ ]:
# --- Single-condition LOSO results (for comparison with fusion) -------------
import pandas as pd

solo_rows = []
for key, X, y, g in [('SSAEP', Xa_r, ya, ga), ('SSVEP', Xb_r, yb, gb)]:
    df, _ = run_all_models(X, y, g)
    df.insert(0, 'condition', key)
    solo_rows.append(df)
solo = pd.concat(solo_rows, ignore_index=True)
print('INDIVIDUAL (single-condition LOSO):')
print(solo[['condition', 'model', 'window_auc', 'subject_auc',
            'subject_accuracy', 'subject_recall']].to_string(index=False))


## 6. Late fusion over the 8-model zoo

In [ ]:
# --- Late fusion over the 8-model zoo ---------------------------------------
fus_rows = []
for name, builder in MODEL_ZOO.items():
    r = evaluate_late_fusion(builder, Xa_r, ya, ga, Xb_r, yb, gb)
    fus_rows.append({
        'model': name,
        'subject_auc': r['subject_auc'],
        'subject_accuracy': r['subject_accuracy'],
        'subject_precision': r['subject_precision'],
        'subject_recall': r['subject_recall'],
        'subject_f1': r['subject_f1'],
        'n_subjects': r['n_subjects'],
        'n_migraine': r['n_migraine'],
    })
fusion = pd.DataFrame(fus_rows)
print(fusion[['model', 'subject_auc', 'subject_accuracy', 'subject_recall',
              'subject_f1', 'n_subjects']].to_string(index=False))


## 7. Fused vs single-condition comparison

In [ ]:
# --- Fused vs. each single-condition AUC ------------------------------------
wide = fusion[['model', 'subject_auc']].rename(columns={'subject_auc': 'fused_AUC'})
l1 = solo[solo['condition'] == 'SSAEP'][['model', 'subject_auc']].rename(
    columns={'subject_auc': 'SSAEP_AUC'})
l2 = solo[solo['condition'] == 'SSVEP'][['model', 'subject_auc']].rename(
    columns={'subject_auc': 'SSVEP_AUC'})
comparison = wide.merge(l1, on='model').merge(l2, on='model')
print(comparison.to_string(index=False))


## 8. ROC curves (late fusion)

In [ ]:
# --- ROC curves for the late-fusion models ----------------------------------
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 6))
for name, builder in MODEL_ZOO.items():
    r = evaluate_late_fusion(builder, Xa_r, ya, ga, Xb_r, yb, gb)
    fpr, tpr = r['roc']
    plt.plot(fpr, tpr, lw=2, label=f'{name} (AUC={r["subject_auc"]:.3f})')
plt.plot([0, 1], [0, 1], 'k--', lw=1, label='Chance')
plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
plt.title('SSAEP+SSVEP late fusion - subject-level ROC (LOSO)')
plt.legend(loc='lower right', fontsize=8); plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'ssaep_ssvep_fusion_roc.png', dpi=150)
plt.show()


## 9. Save results

In [ ]:
# --- Save fusion + comparison tables ----------------------------------------
fusion.to_csv(OUTPUT_DIR / 'ssaep_ssvep_late_fusion_results.csv', index=False)
comparison.to_csv(OUTPUT_DIR / 'ssaep_ssvep_conditions_comparison.csv', index=False)
print('Saved fusion + comparison results to', OUTPUT_DIR)


## Notes
- **Subject-level** metrics come from Leave-One-Subject-Out (LOSO).
- **Window-level AUC** uses grouped 5-fold cross-validation (each test fold holds
  out whole subjects). Per-subject LOSO test folds are single-class, so window
  AUC is not defined there.
- Accuracy / precision / recall / F1 are computed at the Youden-optimal threshold
  for each model's subject-level score.
- Windows are always split by subject, so there is no per-epoch leakage.
- This is a **subject-level feasibility pilot on a small cohort**, not clinical
  generalization. Nested CV / external validation are required for any claim.
